# vidsum on Kaggle GPU

Before running:
1. **Settings → Accelerator: GPU T4 x2** (or T4). Recent PyTorch builds may not support the older P100;
   cell 1 checks this.
2. **Settings → Internet: On** (needed for `git clone`).
3. **Add Input → your dataset** made from `dist/vidsum-data.zip` (`bash scripts/pack_kaggle_data.sh` locally).
4. Private repo: **Add-ons → Secrets → `GITHUB_TOKEN`** (a fine-grained token with read access to the repo).

Outputs go to `/kaggle/working/experiments`. Use **Save Version → Save & Run All** for long runs, so they
continue after you close the tab.

In [ ]:
# 1. GPU check
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    x = torch.randn(256, 256, device="cuda")
    (x @ x).sum().item()  # raises "no kernel image" on GPUs this PyTorch build does not support
    print("GPU kernel test ok")
else:
    print("No GPU: enable Settings -> Accelerator -> GPU T4")

In [ ]:
# 2. Get the code
REPO = "github.com/<your-username>/vidsum.git"   # <- edit
BRANCH = "main"

import os, subprocess
url = f"https://{REPO}"
try:
    from kaggle_secrets import UserSecretsClient
    url = f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@{REPO}"
except Exception:
    pass  # public repo, or no secret attached
os.chdir("/kaggle/working")
if os.path.exists("vidsum"):
    subprocess.run(["git", "-C", "vidsum", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, "vidsum"], check=True)
os.chdir("/kaggle/working/vidsum")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 3. Point vidsum at the data (found automatically under /kaggle/input) and set up the environment
import glob, os, sys
hits = glob.glob("/kaggle/input/**/official/splits/tvsum_splits.json", recursive=True)
assert hits, "Dataset not found: Add Input -> the dataset built from dist/vidsum-data.zip"
DATA = hits[0].rsplit("/official/", 1)[0]

os.environ["VIDSUM_DATA"] = DATA
os.environ["VIDSUM_EXPERIMENTS"] = "/kaggle/working/experiments"
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"   # /kaggle/input is read-only
os.environ["PYTHONPATH"] = "/kaggle/working/vidsum/src"
os.environ["PYTHON"] = sys.executable
sys.path.insert(0, "/kaggle/working/vidsum/src")
print("VIDSUM_DATA =", DATA)
for p in sorted(glob.glob(f"{DATA}/**/*.h5", recursive=True)):
    print(f"  {os.path.getsize(p) / 1e6:7.1f} MB  {p[len(DATA) + 1:]}")

In [ ]:
# 4. Sanity: tests + a 3-epoch smoke run with audio (should print modalities=['googlenet', 'audio_pann'] and device=cuda)
!pip install -q pyyaml
!python -m pytest -q tests
!SIGMAS="0.6" SPLITS="0" EPOCHS=3 EXP=smoke bash scripts/run_splits.sh configs/va_concat.yaml tvsum
!rm -rf /kaggle/working/experiments/smoke

## Experiments
Each sweep is 5 sigmas x 5 splits x 400 epochs. Run the cells you need. Results print at the end of each
cell and are saved to `experiments/<exp>/<dataset>/results.json`.

In [ ]:
# Phase 1 baseline: visual only (original CA-SUM)
!bash scripts/run_splits.sh configs/casum_googlenet.yaml tvsum

In [ ]:
# Phase 3: visual + audio
!bash scripts/run_splits.sh configs/va_concat.yaml tvsum

In [ ]:
# Optional: SumMe and the other fusions
# !bash scripts/run_splits.sh configs/casum_googlenet.yaml summe
# !bash scripts/run_splits.sh configs/va_concat.yaml summe
# !bash scripts/run_splits.sh configs/va_xattn.yaml tvsum
# !bash scripts/run_splits.sh configs/va_softgate.yaml tvsum

In [ ]:
# MoSu subset: visual only vs visual+audio+text (the main tri-modal comparison)
!bash scripts/run_splits.sh configs/mosu_visual.yaml mosu
!bash scripts/run_splits.sh configs/mosu_vat_concat.yaml mosu
# !bash scripts/run_splits.sh configs/mosu_va_concat.yaml mosu
# !bash scripts/run_splits.sh configs/mosu_vat_xattn.yaml mosu
# !bash scripts/run_splits.sh configs/mosu_vat_softgate.yaml mosu

In [ ]:
# Summary table of every finished experiment
import json, glob
for path in sorted(glob.glob("/kaggle/working/experiments/*/*/results.json")):
    exp, dataset = path.split("/")[-3:-1]
    s = json.load(open(path))["summary"]
    print(f"{exp:20s} {dataset:6s} " + "  ".join(f"{m}={v['mean']:.3f}±{v['std']:.3f}" for m, v in s.items()))

In [ ]:
# Download: results + configs + histories (scores per epoch are large; add them if you need curves)
!cd /kaggle/working && zip -q -r results.zip experiments -i '*/results.json' '*/config.json' '*/history.json' && ls -lh results.zip